In [ ]:
# read & manipulate data
import pandas as pd 
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
import joblib
from sklearn.model_selection import train_test_split

# visualisations
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid', context='notebook')

# misc
import random as rn
from datetime import datetime
import os

# manual parameters
RANDOM_SEED = 42
VALIDATE_SIZE = 0.2

# setting random seeds for libraries to ensure reproducibility
np.random.seed(RANDOM_SEED)
rn.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Define patients and their corresponding files
patients = {
    1: ['/kaggle/input/control-set-all/organized_fcs_data1a.csv',
        '/kaggle/input/control-set-all/organized_fcs_data1b.csv',
        '/kaggle/input/control-set-all/organized_fcs_data1c.csv'],
    2: ['/kaggle/input/control-set-all/organized_fcs_data2a.csv',
        '/kaggle/input/control-set-all/organized_fcs_data2b.csv',
        '/kaggle/input/control-set-all/organized_fcs_data2c.csv'],
    3: ['/kaggle/input/control-set-all/organized_fcs_data3a.csv',
        '/kaggle/input/control-set-all/organized_fcs_data3b.csv',
        '/kaggle/input/control-set-all/organized_fcs_data3c.csv'],
    4: ['/kaggle/input/control-set-all/organized_fcs_data4a.csv',
        '/kaggle/input/control-set-all/organized_fcs_data4b.csv',
        '/kaggle/input/control-set-all/organized_fcs_data4c.csv'],
    5: ['/kaggle/input/control-set-all/organized_fcs_data5a.csv',
        '/kaggle/input/control-set-all/organized_fcs_data5b.csv',
        '/kaggle/input/control-set-all/organized_fcs_data5c.csv'],
    6: ['/kaggle/input/control-set-all/organized_fcs_data6a.csv',
        '/kaggle/input/control-set-all/organized_fcs_data6b.csv',
        '/kaggle/input/control-set-all/organized_fcs_data6c.csv']
}

# Load, merge, and sample data for each patient
sampled_dfs = []

for patient_id, file_paths in patients.items():
    # Load all three CSV files for the patient and drop the "Time" column
    patient_data = pd.concat([pd.read_csv(file).drop(columns=['Time'], errors='ignore') for file in file_paths])
    sampled_dfs.append(patient_data)

# Concatenate all patient data
full_dataset = pd.concat(sampled_dfs)

# Shuffle the data
shuffled_dataset = full_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into training (80%) and validation (20%)
train_df, val_df = train_test_split(shuffled_dataset, test_size=0.2, random_state=42)

# Save the splits
train_df.to_csv('training_data.csv', index=False)
val_df.to_csv('validation_data.csv', index=False)

# Print summary
print(f"Total dataset size: {len(full_dataset)} cells")
print(f"Training set size: {len(train_df)} cells")
print(f"Validation set size: {len(val_df)} cells")

# configure our pipeline
pipeline = Pipeline([('scaler', MinMaxScaler())])
pipeline.fit(train_df)

# Save the pipeline to a file
joblib.dump(pipeline, 'April_pipeline.pkl')

# transform the training and validation data with these parameters
X_train_transformed = pipeline.transform(train_df)
X_validate_transformed = pipeline.transform(val_df)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_transformed).to(device)
X_val_tensor = torch.FloatTensor(X_validate_transformed).to(device)

# Create DataLoaders
BATCH_SIZE = 256
train_dataset = TensorDataset(X_train_tensor, X_train_tensor)
val_dataset = TensorDataset(X_val_tensor, X_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Define the Autoencoder model
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ELU(),
            nn.Linear(input_dim, 16),
            nn.ELU(),
            nn.Linear(16, 8),
            nn.ELU(),
            nn.Linear(8, 4),
            nn.ELU(),
            nn.Linear(4, 2),
            nn.ELU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(2, 4),
            nn.ELU(),
            nn.Linear(4, 8),
            nn.ELU(),
            nn.Linear(8, 16),
            nn.ELU(),
            nn.Linear(16, input_dim),
            nn.ELU()
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# data dimensions // hyperparameters 
input_dim = X_train_transformed.shape[1]
EPOCHS = 100

autoencoder= Autoencoder(input_dim=input_dim)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters())

# Print model summary
print(autoencoder)
print(f"Total parameters: {sum(p.numel() for p in autoencoder.parameters())}")

# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy (for reconstruction tasks, this is approximate)
        pred = output.round()
        correct += pred.eq(target.round()).sum().item()
        total += target.numel()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

# Validation function
def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item()
            
            # Calculate accuracy
            pred = output.round()
            correct += pred.eq(target.round()).sum().item()
            total += target.numel()
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

# Early stopping class
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0001, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_loss = None
        self.counter = 0
        self.best_weights = None
        
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            if self.restore_best_weights:
                self.best_weights = model.state_dict().copy()
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            if self.restore_best_weights:
                self.best_weights = model.state_dict().copy()
        else:
            self.counter += 1
            
        if self.counter >= self.patience:
            if self.restore_best_weights and self.best_weights is not None:
                model.load_state_dict(self.best_weights)
            return True
        return False

# Initialize early stopping
early_stopping = EarlyStopping(patience=10, min_delta=0.0001, restore_best_weights=True)

# Create logs directory
yyyymmddHHMM = datetime.now().strftime('%Y%m%d%H%M')
log_subdir = f'{yyyymmddHHMM}_batch{BATCH_SIZE}_layers{len(list(autoencoder.modules()))}'
os.makedirs('logs', exist_ok=True)

# Training loop
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

best_val_loss = float('inf')

print("Starting training...")
for epoch in range(EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(autoencoder, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate_epoch(autoencoder, val_loader, criterion, device)
    
    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(autoencoder.state_dict(), 'autoencoder_best_weights_April_AE.pth')
    
    print(f'Epoch [{epoch+1}/{EPOCHS}], '
          f'Train Loss: {train_loss:.6f}, Train Acc: {train_acc:.2f}%, '
          f'Val Loss: {val_loss:.6f}, Val Acc: {val_acc:.2f}%')
    
    # Early stopping
    if early_stopping(val_loss, autoencoder):
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

print("Training completed!")

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy (%)')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

# Save final model
torch.save(autoencoder.state_dict(), 'autoencoder_final_April_AE.pth')
torch.save(autoencoder, 'autoencoder_complete_April_AE.pth')

print(f"Models saved:")
print(f"- Best weights: autoencoder_best_weights_April_AE.pth")
print(f"- Final model: autoencoder_final_April_AE.pth")
print(f"- Complete model: autoencoder_complete_April_AE.pth")